# MADAD — 03 Tuned XGBoost Quantile Forecasting
## P10 / P50 / P90

Standalone XGBoost notebook using the same leakage-safe features and chronological split.

- P50 is tuned for median demand forecasting.
- P90 is tuned separately.
- P10 is trained with the established initial quantile configuration.
- Saves tuning tables, metrics, predictions, and best models.

**Test remains untouched.**


In [1]:
# ============================================================
# 1) Mount Drive and set paths
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE_PATH = Path(
    '/content/drive/MyDrive/Madad for Medical Logistics/Dataset/Dryad_Dataset'
)

DATA_PATH = BASE_PATH / 'Madad_Integrated_Data.csv'

OUTPUT_PATH = (
    BASE_PATH
    / 'forecasting_outputs_clean'
    / '03_xgboost_quantile'
)

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print('Data path:', DATA_PATH)
print('Exists:', DATA_PATH.exists())
print('Save folder:', OUTPUT_PATH)

# ============================================================
# 2) Install and import packages
# ============================================================
!pip -q install "xgboost>=2.0" joblib

import warnings
warnings.filterwarnings('ignore')

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_pinball_loss,
    r2_score
)

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

RANDOM_STATE = 42
EPS = 1e-6


Mounted at /content/drive
Data path: /content/drive/MyDrive/Madad for Medical Logistics/Dataset/Dryad_Dataset/Madad_Integrated_Data.csv
Exists: True
Save folder: /content/drive/MyDrive/Madad for Medical Logistics/Dataset/Dryad_Dataset/forecasting_outputs_clean/03_xgboost_quantile


In [2]:
# ============================================================
# 3) Load integrated Madad data
# ============================================================
df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

df['date_parsed'] = pd.to_datetime(
    df['date_parsed'],
    errors='coerce'
)

group_cols = ['hf_pk', 'productID']

df_model = (
    df
    .dropna(
        subset=[
            'date_parsed',
            'hf_pk',
            'productID',
            'consumption'
        ]
    )
    .copy()
)

df_model = (
    df_model
    .sort_values(group_cols + ['date_parsed'])
    .reset_index(drop=True)
)

print('Integrated rows:', f'{len(df_model):,}')
print('Facilities:', df_model['hf_pk'].nunique())
print('Products:', df_model['productID'].nunique())
print(
    'Date range:',
    df_model['date_parsed'].min(),
    'to',
    df_model['date_parsed'].max()
)

print('\nConsumption summary:')
print(df_model['consumption'].describe())


Integrated rows: 457,225
Facilities: 1091
Products: 36
Date range: 2019-10-01 00:00:00 to 2023-11-01 00:00:00

Consumption summary:
count    457225.000000
mean        161.034016
std         378.274544
min           0.000000
25%          10.000000
50%          50.000000
75%         173.000000
max       48000.000000
Name: consumption, dtype: float64


In [3]:
# ============================================================
# 4) Calendar-safe helper functions
# ============================================================
def calendar_month_gap(current_date, previous_date):
    return (
        (current_date.dt.year - previous_date.dt.year) * 12
        + (current_date.dt.month - previous_date.dt.month)
    )


def add_calendar_safe_lags(
    data,
    group_cols,
    date_col,
    value_cols,
    lags
):
    data = data.copy()

    for col in value_cols:
        if col not in data.columns:
            continue

        grouped = data.groupby(
            group_cols,
            sort=False
        )

        for lag in lags:
            previous_value = grouped[col].shift(lag)
            previous_date = grouped[date_col].shift(lag)

            gap = calendar_month_gap(
                data[date_col],
                previous_date
            )

            data[f'{col}_lag_{lag}'] = (
                previous_value.where(gap == lag)
            )

    return data


def add_calendar_lagged_aggregate_features(
    base_df,
    group_keys,
    source_col,
    prefix,
    lags=range(1, 7),
    agg='mean'
):
    # Monthly group aggregate is built using only that month's rows.
    monthly = (
        base_df
        .groupby(
            group_keys + ['date_parsed'],
            as_index=False
        )[source_col]
        .agg(agg)
        .rename(columns={source_col: '_aggregate_value'})
    )

    result = base_df

    for lag in lags:
        lagged = monthly.copy()

        lagged['date_parsed'] = (
            lagged['date_parsed']
            + pd.offsets.MonthBegin(lag)
        )

        lagged = lagged.rename(
            columns={
                '_aggregate_value':
                f'{prefix}_lag_{lag}'
            }
        )

        result = result.merge(
            lagged[
                group_keys
                + ['date_parsed', f'{prefix}_lag_{lag}']
            ],
            on=group_keys + ['date_parsed'],
            how='left'
        )

    return result


In [4]:
# ============================================================
# 5) Local temporal features BEFORE target construction
# ============================================================
lag_source_cols = [
    c for c in [
        'consumption',
        'received',
        'openBalance',
        'closeBalance',
        'stockout'
    ]
    if c in df_model.columns
]

df_model = add_calendar_safe_lags(
    data=df_model,
    group_cols=group_cols,
    date_col='date_parsed',
    value_cols=lag_source_cols,
    lags=range(1, 13)
)

for col in [
    'consumption',
    'received',
    'closeBalance'
]:
    if col not in df_model.columns:
        continue

    for window, min_history in [
        (3, 2),
        (6, 3),
        (12, 6)
    ]:
        lag_cols = [
            f'{col}_lag_{lag}'
            for lag in range(1, window + 1)
            if f'{col}_lag_{lag}' in df_model.columns
        ]

        history_count = (
            df_model[lag_cols]
            .notna()
            .sum(axis=1)
        )

        df_model[
            f'{col}_history_count_{window}'
        ] = history_count

        df_model[
            f'{col}_roll_mean_{window}'
        ] = (
            df_model[lag_cols]
            .mean(axis=1, skipna=True)
            .where(history_count >= min_history)
        )

        df_model[
            f'{col}_roll_std_{window}'
        ] = (
            df_model[lag_cols]
            .std(axis=1, skipna=True)
            .where(history_count >= min_history)
        )

for window, min_history in [
    (3, 2),
    (6, 3),
    (12, 6)
]:
    lag_cols = [
        f'stockout_lag_{lag}'
        for lag in range(1, window + 1)
        if f'stockout_lag_{lag}' in df_model.columns
    ]

    history_count = (
        df_model[lag_cols]
        .notna()
        .sum(axis=1)
    )

    df_model[f'stockout_rate_past_{window}'] = (
        df_model[lag_cols]
        .mean(axis=1, skipna=True)
        .where(history_count >= min_history)
    )

df_model['year'] = df_model['date_parsed'].dt.year
df_model['month'] = df_model['date_parsed'].dt.month
df_model['quarter_num'] = df_model['date_parsed'].dt.quarter

df_model['month_sin'] = np.sin(
    2 * np.pi * df_model['month'] / 12
)

df_model['month_cos'] = np.cos(
    2 * np.pi * df_model['month'] / 12
)

if {
    'consumption_roll_mean_3',
    'consumption_roll_mean_6'
}.issubset(df_model.columns):
    df_model['demand_trend_3_vs_6'] = (
        df_model['consumption_roll_mean_3']
        - df_model['consumption_roll_mean_6']
    )

if {
    'consumption_roll_mean_3',
    'consumption_roll_mean_12'
}.issubset(df_model.columns):
    df_model['demand_trend_3_vs_12'] = (
        df_model['consumption_roll_mean_3']
        - df_model['consumption_roll_mean_12']
    )

if 'consumption_roll_mean_12' in df_model.columns:
    df_model['current_consumption_vs_12m'] = (
        df_model['consumption']
        / (
            df_model['consumption_roll_mean_12'].abs()
            + EPS
        )
    )

if {
    'closeBalance',
    'consumption_roll_mean_3'
}.issubset(df_model.columns):
    df_model['inventory_runway_3m'] = (
        df_model['closeBalance']
        / (
            df_model['consumption_roll_mean_3'].abs()
            + EPS
        )
    )

if {
    'received',
    'consumption'
}.issubset(df_model.columns):
    df_model['received_to_consumption'] = (
        df_model['received']
        / (
            df_model['consumption'].abs()
            + EPS
        )
    )

df_model['facility_product_observation_count_prior'] = (
    df_model
    .groupby(group_cols)
    .cumcount()
)

print('Local temporal features complete.')


Local temporal features complete.


In [5]:
# ============================================================
# 6) NEW V2 CONTEXT:
# Lagged product-level and district-product demand signals
# ============================================================
# These are all shifted forward by explicit calendar months.
# Therefore, a current row only receives aggregate demand
# information from prior calendar months.

df_model = add_calendar_lagged_aggregate_features(
    base_df=df_model,
    group_keys=['productID'],
    source_col='consumption',
    prefix='product_mean_consumption',
    lags=range(1, 7),
    agg='mean'
)

if 'district' in df_model.columns:
    df_model = add_calendar_lagged_aggregate_features(
        base_df=df_model,
        group_keys=['district', 'productID'],
        source_col='consumption',
        prefix='district_product_mean_consumption',
        lags=range(1, 7),
        agg='mean'
    )

for prefix in [
    'product_mean_consumption',
    'district_product_mean_consumption'
]:
    lag3 = [
        f'{prefix}_lag_{lag}'
        for lag in range(1, 4)
        if f'{prefix}_lag_{lag}' in df_model.columns
    ]

    lag6 = [
        f'{prefix}_lag_{lag}'
        for lag in range(1, 7)
        if f'{prefix}_lag_{lag}' in df_model.columns
    ]

    if lag3:
        df_model[f'{prefix}_roll3'] = (
            df_model[lag3]
            .mean(axis=1, skipna=True)
        )

    if lag6:
        df_model[f'{prefix}_roll6'] = (
            df_model[lag6]
            .mean(axis=1, skipna=True)
        )

    if (
        f'{prefix}_roll3' in df_model.columns
        and f'{prefix}_roll6' in df_model.columns
    ):
        df_model[f'{prefix}_trend_3_vs_6'] = (
            df_model[f'{prefix}_roll3']
            - df_model[f'{prefix}_roll6']
        )

print('Contextual aggregate features complete.')


Contextual aggregate features complete.


In [6]:
# ============================================================
# EXPERIMENT 2 — Demand-dynamics features
# ============================================================
def safe_ratio(num, den):
    return num / (np.abs(den) + EPS)

if 'consumption_lag_1' in df_model.columns:
    df_model['consumption_growth_1m'] = df_model['consumption'] - df_model['consumption_lag_1']
    df_model['consumption_growth_rate_1m'] = safe_ratio(df_model['consumption'] - df_model['consumption_lag_1'], df_model['consumption_lag_1'])

if {'consumption_roll_mean_3','consumption_roll_mean_6'}.issubset(df_model.columns):
    df_model['recent_vs_6m_ratio'] = safe_ratio(df_model['consumption_roll_mean_3'], df_model['consumption_roll_mean_6'])

if {'consumption_roll_mean_3','consumption_roll_mean_12'}.issubset(df_model.columns):
    df_model['recent_vs_12m_ratio'] = safe_ratio(df_model['consumption_roll_mean_3'], df_model['consumption_roll_mean_12'])

for window in [3,6,12]:
    mean_col=f'consumption_roll_mean_{window}'
    std_col=f'consumption_roll_std_{window}'
    if {mean_col,std_col}.issubset(df_model.columns):
        df_model[f'consumption_cv_{window}m'] = safe_ratio(df_model[std_col], df_model[mean_col])
    lag_cols=[f'consumption_lag_{lag}' for lag in range(1,window+1) if f'consumption_lag_{lag}' in df_model.columns]
    if lag_cols:
        valid_count=df_model[lag_cols].notna().sum(axis=1)
        zero_count=df_model[lag_cols].eq(0).sum(axis=1)
        df_model[f'zero_consumption_rate_{window}m'] = zero_count / valid_count.replace(0,np.nan)

if 'consumption_roll_mean_6' in df_model.columns:
    df_model['demand_spike_ratio'] = safe_ratio(df_model['consumption'], df_model['consumption_roll_mean_6'])

if {'consumption_lag_1','consumption_lag_3'}.issubset(df_model.columns):
    df_model['consumption_momentum_3m'] = df_model['consumption_lag_1'] - df_model['consumption_lag_3']

print('Experiment 2 demand-dynamics features added.')

Experiment 2 demand-dynamics features added.


In [7]:
# ============================================================
# 7) Construct NEXT-CALENDAR-MONTH demand target
# ============================================================
grouped = df_model.groupby(
    group_cols,
    sort=False
)

df_model['next_date_parsed'] = (
    grouped['date_parsed']
    .shift(-1)
)

df_model['next_month_consumption_observed'] = (
    grouped['consumption']
    .shift(-1)
)

df_model['month_start'] = (
    df_model['date_parsed']
    .dt.to_period('M')
    .dt.to_timestamp()
)

df_model['next_month_start'] = (
    df_model['next_date_parsed']
    .dt.to_period('M')
    .dt.to_timestamp()
)

df_model['expected_next_month'] = (
    df_model['month_start']
    + pd.offsets.MonthBegin(1)
)

is_true_next_month = (
    df_model['next_month_start']
    == df_model['expected_next_month']
)

df_model['target_consumption_next_month'] = (
    np.where(
        is_true_next_month,
        df_model['next_month_consumption_observed'],
        np.nan
    )
)

df_model = (
    df_model
    .dropna(
        subset=['target_consumption_next_month']
    )
    .copy()
)

target_gap = calendar_month_gap(
    df_model['next_month_start'],
    df_model['month_start']
)

assert (target_gap == 1).all()

print(
    'Eligible forecasting rows:',
    f'{len(df_model):,}'
)

print(
    'Target relation:',
    'ALL rows are exactly next calendar month.'
)

print('\nTarget summary:')
print(
    df_model[
        'target_consumption_next_month'
    ].describe()
)


Eligible forecasting rows: 313,698
Target relation: ALL rows are exactly next calendar month.

Target summary:
count    313698.000000
mean        167.782345
std         400.037253
min           0.000000
25%          10.000000
50%          50.000000
75%         195.000000
max       48000.000000
Name: target_consumption_next_month, dtype: float64


In [8]:
# ============================================================
# 8) Chronological whole-month split
# ============================================================
df_model = (
    df_model
    .sort_values('date_parsed')
    .reset_index(drop=True)
)

unique_dates = np.array(
    sorted(
        df_model[
            'date_parsed'
        ].dropna().unique()
    )
)

n_dates = len(unique_dates)

train_cut = (
    max(
        1,
        int(n_dates * 0.715)
    )
    - 1
)

val_cut = (
    max(
        train_cut + 1,
        int(
            n_dates
            * (0.715 + 0.127)
        )
    )
    - 1
)

train_end_date = pd.Timestamp(
    unique_dates[train_cut]
)

val_end_date = pd.Timestamp(
    unique_dates[
        min(
            val_cut,
            n_dates - 1
        )
    ]
)

train_df = df_model[
    df_model['date_parsed']
    <= train_end_date
].copy()

val_df = df_model[
    (
        df_model['date_parsed']
        > train_end_date
    )
    & (
        df_model['date_parsed']
        <= val_end_date
    )
].copy()

test_df = df_model[
    df_model['date_parsed']
    > val_end_date
].copy()

print(
    'Train:',
    f'{len(train_df):,}',
    train_df['date_parsed'].min(),
    'to',
    train_df['date_parsed'].max()
)

print(
    'Validation:',
    f'{len(val_df):,}',
    val_df['date_parsed'].min(),
    'to',
    val_df['date_parsed'].max()
)

print(
    'Test:',
    f'{len(test_df):,}',
    test_df['date_parsed'].min(),
    'to',
    test_df['date_parsed'].max()
)

print(
    'Total:',
    f'{len(train_df) + len(val_df) + len(test_df):,}'
)

assert (
    train_df['date_parsed'].max()
    < val_df['date_parsed'].min()
)

assert (
    val_df['date_parsed'].max()
    < test_df['date_parsed'].min()
)


Train: 211,452 2019-10-01 00:00:00 to 2022-08-01 00:00:00
Validation: 39,864 2022-09-01 00:00:00 to 2023-02-01 00:00:00
Test: 62,382 2023-03-01 00:00:00 to 2023-10-01 00:00:00
Total: 313,698


In [9]:
# ============================================================
# 9) Evaluation helpers
# ============================================================
TARGET = 'target_consumption_next_month'

def regression_metrics(
    y_true,
    y_pred
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    denominator = np.sum(
        np.abs(y_true)
    )

    wape = (
        np.sum(
            np.abs(
                y_true - y_pred
            )
        )
        / denominator
        if denominator > 0
        else np.nan
    )

    return {
        'MAE': mae,
        'RMSE': rmse,
        'WAPE': wape,
        'R2': r2
    }


def print_metrics(
    name,
    y_true,
    y_pred
):
    metrics = regression_metrics(
        y_true,
        y_pred
    )

    print(f'--- {name} ---')
    print(
        f"MAE : "
        f"{metrics['MAE']:.4f}"
    )
    print(
        f"RMSE: "
        f"{metrics['RMSE']:.4f}"
    )
    print(
        f"WAPE: "
        f"{metrics['WAPE']*100:.2f}%"
    )
    print(
        f"R²  : "
        f"{metrics['R2']:.4f}"
    )

    return metrics


In [10]:
# ============================================================
# 11) Select leakage-safe forecasting features
# ============================================================
base_numeric = [
    'consumption_growth_1m',
    'consumption_growth_rate_1m',
    'recent_vs_6m_ratio',
    'recent_vs_12m_ratio',
    'consumption_cv_3m',
    'consumption_cv_6m',
    'consumption_cv_12m',
    'zero_consumption_rate_3m',
    'zero_consumption_rate_6m',
    'zero_consumption_rate_12m',
    'demand_spike_ratio',
    'consumption_momentum_3m',

    'consumption',
    'received',
    'openBalance',
    'closeBalance',
    'normAvg',
    'normStd',
    'year',
    'month',
    'quarter_num',
    'month_sin',
    'month_cos',
    'facility_product_observation_count_prior',
    'demand_trend_3_vs_6',
    'demand_trend_3_vs_12',
    'current_consumption_vs_12m',
    'inventory_runway_3m',
    'received_to_consumption'
]

dynamic_features = [
    c for c in df_model.columns
    if any(
        c.startswith(prefix)
        for prefix in [
            'consumption_lag_',
            'received_lag_',
            'openBalance_lag_',
            'closeBalance_lag_',
            'stockout_lag_',
            'consumption_roll_',
            'received_roll_',
            'closeBalance_roll_',
            'consumption_history_count_',
            'received_history_count_',
            'closeBalance_history_count_',
            'stockout_rate_past_',
            'product_mean_consumption_',
            'district_product_mean_consumption_'
        ]
    )
]

numeric_features = [
    c
    for c in (
        base_numeric
        + dynamic_features
    )
    if c in train_df.columns
]

numeric_features = list(
    dict.fromkeys(
        numeric_features
    )
)

categorical_features = [
    c for c in [
        'hf_pk',
        'productID',
        'facility_type',
        'district'
    ]
    if c in train_df.columns
]

numeric_features = [
    c
    for c in numeric_features
    if not train_df[c].isna().all()
]

feature_cols = (
    numeric_features
    + categorical_features
)

print(
    'Numeric features:',
    len(numeric_features)
)

print(
    'Categorical features:',
    len(categorical_features)
)

print(
    'Total raw features:',
    len(feature_cols)
)


Numeric features: 137
Categorical features: 4
Total raw features: 141


In [11]:
# ============================================================
# 12) Preprocessing
# ============================================================
numeric_pipeline = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(
                strategy='median'
            )
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(
                strategy='most_frequent'
            )
        ),
        (
            'onehot',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=True
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            numeric_pipeline,
            numeric_features
        ),
        (
            'cat',
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder='drop'
)

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]

y_train = train_df[TARGET].values
y_val = val_df[TARGET].values

X_train_t = (
    preprocessor
    .fit_transform(X_train)
)

X_val_t = (
    preprocessor
    .transform(X_val)
)

print(
    'Transformed Train shape:',
    X_train_t.shape
)

print(
    'Transformed Validation shape:',
    X_val_t.shape
)


Transformed Train shape: (211452, 1282)
Transformed Validation shape: (39864, 1282)


In [12]:
# ============================================================
# XGBOOST P10 — Established lower-quantile configuration
# ============================================================
p10_model = XGBRegressor(
    objective='reg:quantileerror',
    quantile_alpha=0.10,
    n_estimators=600,
    max_depth=5,
    learning_rate=0.03,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=2.0,
    random_state=RANDOM_STATE,
    tree_method='hist',
    device='cuda',
    n_jobs=-1
)

p10_model.fit(
    X_train_t,
    y_train,
    verbose=False
)

p10_pred = np.clip(
    p10_model.predict(X_val_t),
    0,
    None
)

p10_pinball = mean_pinball_loss(
    y_val,
    p10_pred,
    alpha=0.10
)

print('P10 Pinball:', round(p10_pinball, 4))


P10 Pinball: 15.8229


In [13]:
# ============================================================
# EXPERIMENT 2A — P50 tuning
# ============================================================
p50_candidates = [
    {'max_depth':4,'learning_rate':0.03,'min_child_weight':5,'subsample':0.90,'colsample_bytree':0.90,'reg_alpha':0.10,'reg_lambda':2.0},
    {'max_depth':5,'learning_rate':0.03,'min_child_weight':5,'subsample':0.90,'colsample_bytree':0.90,'reg_alpha':0.10,'reg_lambda':2.0},
    {'max_depth':6,'learning_rate':0.03,'min_child_weight':5,'subsample':0.90,'colsample_bytree':0.90,'reg_alpha':0.10,'reg_lambda':2.0},
    {'max_depth':5,'learning_rate':0.02,'min_child_weight':3,'subsample':0.90,'colsample_bytree':0.90,'reg_alpha':0.10,'reg_lambda':2.0},
    {'max_depth':5,'learning_rate':0.05,'min_child_weight':10,'subsample':0.85,'colsample_bytree':0.90,'reg_alpha':0.20,'reg_lambda':3.0},
    {'max_depth':6,'learning_rate':0.02,'min_child_weight':10,'subsample':0.85,'colsample_bytree':0.85,'reg_alpha':0.20,'reg_lambda':3.0}
]
p50_rows=[]
p50_models={}
for i,params in enumerate(p50_candidates,1):
    print(f'P50 candidate {i}/{len(p50_candidates)}')
    model=XGBRegressor(objective='reg:quantileerror',quantile_alpha=0.50,n_estimators=1200,early_stopping_rounds=60,eval_metric='quantile',random_state=RANDOM_STATE,tree_method='hist',device='cuda',n_jobs=-1,**params)
    model.fit(X_train_t,y_train,eval_set=[(X_val_t,y_val)],verbose=False)
    pred=np.clip(model.predict(X_val_t),0,None)
    m=regression_metrics(y_val,pred)
    loss=mean_pinball_loss(y_val,pred,alpha=0.50)
    p50_rows.append({'candidate':i,'pinball_loss':loss,'best_iteration':model.best_iteration,**m,**params})
    p50_models[i]=model
p50_search_results=pd.DataFrame(p50_rows).sort_values(['pinball_loss','MAE']).reset_index(drop=True)
display(p50_search_results)
best_p50_id=int(p50_search_results.iloc[0]['candidate'])
best_p50_model=p50_models[best_p50_id]
best_p50_pred=np.clip(best_p50_model.predict(X_val_t),0,None)
best_p50_metrics=regression_metrics(y_val,best_p50_pred)
best_p50_pinball=mean_pinball_loss(y_val,best_p50_pred,alpha=0.50)
print('BEST P50:',best_p50_id)
print('MAE:',round(best_p50_metrics['MAE'],4),'| Pinball:',round(best_p50_pinball,4))

P50 candidate 1/6
P50 candidate 2/6
P50 candidate 3/6
P50 candidate 4/6
P50 candidate 5/6
P50 candidate 6/6


,candidate,pinball_loss,best_iteration,MAE,RMSE,WAPE,R2,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda
0,3,47.506186,1199,95.012371,323.910794,0.531455,0.334999,6,0.03,5,0.90,0.90,0.1,2.0
1,5,47.643854,1198,95.287707,323.896886,0.532996,0.335056,5,0.05,10,0.85,0.90,0.2,3.0
2,2,47.702495,1199,95.404990,324.026684,0.533652,0.334523,5,0.03,5,0.90,0.90,0.1,2.0
3,6,47.721375,1199,95.442750,325.423809,0.533863,0.328772,6,0.02,10,0.85,0.85,0.2,3.0
4,4,47.875531,1199,95.751063,325.143585,0.535587,0.329927,5,0.02,3,0.90,0.90,0.1,2.0
5,1,48.057838,1199,96.115675,324.411222,0.537627,0.332943,4,0.03,5,0.90,0.90,0.1,2.0


BEST P50: 3
MAE: 95.0124 | Pinball: 47.5062


In [14]:
# ============================================================
# EXPERIMENT 2B — P90 tuning
# ============================================================
p90_candidates = [
    {'max_depth':4,'learning_rate':0.03,'min_child_weight':5,'subsample':0.90,'colsample_bytree':0.90,'reg_alpha':0.10,'reg_lambda':2.0},
    {'max_depth':5,'learning_rate':0.03,'min_child_weight':5,'subsample':0.90,'colsample_bytree':0.90,'reg_alpha':0.10,'reg_lambda':2.0},
    {'max_depth':6,'learning_rate':0.03,'min_child_weight':5,'subsample':0.90,'colsample_bytree':0.90,'reg_alpha':0.10,'reg_lambda':2.0},
    {'max_depth':5,'learning_rate':0.02,'min_child_weight':10,'subsample':0.85,'colsample_bytree':0.90,'reg_alpha':0.20,'reg_lambda':3.0},
    {'max_depth':6,'learning_rate':0.05,'min_child_weight':10,'subsample':0.85,'colsample_bytree':0.85,'reg_alpha':0.20,'reg_lambda':3.0}
]
p90_rows=[]
p90_models={}
for i,params in enumerate(p90_candidates,1):
    print(f'P90 candidate {i}/{len(p90_candidates)}')
    model=XGBRegressor(objective='reg:quantileerror',quantile_alpha=0.90,n_estimators=1200,early_stopping_rounds=60,eval_metric='quantile',random_state=RANDOM_STATE,tree_method='hist',device='cuda',n_jobs=-1,**params)
    model.fit(X_train_t,y_train,eval_set=[(X_val_t,y_val)],verbose=False)
    pred=np.clip(model.predict(X_val_t),0,None)
    loss=mean_pinball_loss(y_val,pred,alpha=0.90)
    p90_rows.append({'candidate':i,'pinball_loss':loss,'best_iteration':model.best_iteration,**params})
    p90_models[i]=model
p90_search_results=pd.DataFrame(p90_rows).sort_values('pinball_loss').reset_index(drop=True)
display(p90_search_results)
best_p90_id=int(p90_search_results.iloc[0]['candidate'])
best_p90_model=p90_models[best_p90_id]
best_p90_pred=np.clip(best_p90_model.predict(X_val_t),0,None)
best_p90_pinball=mean_pinball_loss(y_val,best_p90_pred,alpha=0.90)
print('BEST P90:',best_p90_id)
print('Pinball:',round(best_p90_pinball,4))

P90 candidate 1/5
P90 candidate 2/5
P90 candidate 3/5
P90 candidate 4/5
P90 candidate 5/5


,candidate,pinball_loss,best_iteration,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda
0,5,36.539719,1193,6,0.05,10,0.85,0.85,0.2,3.0
1,3,36.690023,1199,6,0.03,5,0.90,0.90,0.1,2.0
2,2,36.781231,1199,5,0.03,5,0.90,0.90,0.1,2.0
3,4,37.009908,1199,5,0.02,10,0.85,0.90,0.2,3.0
4,1,37.175738,1199,4,0.03,5,0.90,0.90,0.1,2.0


BEST P90: 5
Pinball: 36.5397


In [15]:
# ============================================================
# XGBOOST — Final Quantile Validation Evaluation
# ============================================================
q = np.sort(
    np.column_stack([
        p10_pred,
        best_p50_pred,
        best_p90_pred
    ]),
    axis=1
)

xgb_p10 = q[:, 0]
xgb_p50 = q[:, 1]
xgb_p90 = q[:, 2]

xgb_p10_pinball = mean_pinball_loss(y_val, xgb_p10, alpha=0.10)
xgb_p50_pinball = mean_pinball_loss(y_val, xgb_p50, alpha=0.50)
xgb_p90_pinball = mean_pinball_loss(y_val, xgb_p90, alpha=0.90)

xgb_p50_metrics = regression_metrics(y_val, xgb_p50)

xgb_coverage = np.mean(
    (y_val >= xgb_p10)
    & (y_val <= xgb_p90)
)

xgb_interval_width = np.mean(
    xgb_p90 - xgb_p10
)

print('XGBoost Quantile Validation Results')
print('P10 Pinball:', round(xgb_p10_pinball, 4))
print('P50 Pinball:', round(xgb_p50_pinball, 4))
print('P90 Pinball:', round(xgb_p90_pinball, 4))
print('P50 MAE:', round(xgb_p50_metrics['MAE'], 4))
print('P50 RMSE:', round(xgb_p50_metrics['RMSE'], 4))
print('P50 WAPE:', f"{xgb_p50_metrics['WAPE']*100:.2f}%")
print('P50 R²:', round(xgb_p50_metrics['R2'], 4))
print('P10–P90 Coverage:', f'{xgb_coverage*100:.2f}%')
print('Mean interval width:', round(xgb_interval_width, 2))


XGBoost Quantile Validation Results
P10 Pinball: 15.8208
P50 Pinball: 47.5054
P90 Pinball: 36.5385
P50 MAE: 95.0108
P50 RMSE: 323.9083
P50 WAPE: 53.14%
P50 R²: 0.335
P10–P90 Coverage: 80.67%
Mean interval width: 267.77


In [16]:
# ============================================================
# SAVE — XGBoost Validation Results
# ============================================================
xgb_summary = pd.DataFrame([
    {
        'Model': 'Tuned XGBoost Quantile',
        'P50_MAE': xgb_p50_metrics['MAE'],
        'P50_RMSE': xgb_p50_metrics['RMSE'],
        'P50_WAPE': xgb_p50_metrics['WAPE'],
        'P50_R2': xgb_p50_metrics['R2'],
        'P10_Pinball': xgb_p10_pinball,
        'P50_Pinball': xgb_p50_pinball,
        'P90_Pinball': xgb_p90_pinball,
        'Coverage_P10_P90': xgb_coverage,
        'Mean_Interval_Width': xgb_interval_width
    }
])

display(xgb_summary)

xgb_summary.to_csv(
    OUTPUT_PATH / 'xgboost_validation_metrics.csv',
    index=False
)

p50_search_results.to_csv(
    OUTPUT_PATH / 'xgboost_p50_tuning_results.csv',
    index=False
)

p90_search_results.to_csv(
    OUTPUT_PATH / 'xgboost_p90_tuning_results.csv',
    index=False
)

pred_df = val_df[
    ['hf_pk', 'productID', 'date_parsed', TARGET]
].copy()

pred_df['P10'] = xgb_p10
pred_df['P50'] = xgb_p50
pred_df['P90'] = xgb_p90

pred_df.to_csv(
    OUTPUT_PATH / 'xgboost_validation_predictions.csv',
    index=False
)

joblib.dump(
    best_p50_model,
    OUTPUT_PATH / 'xgboost_best_p50_model.joblib'
)

joblib.dump(
    best_p90_model,
    OUTPUT_PATH / 'xgboost_best_p90_model.joblib'
)

joblib.dump(
    p10_model,
    OUTPUT_PATH / 'xgboost_p10_model.joblib'
)

print('=' * 72)
print('MADAD — XGBOOST VALIDATION CHECKPOINT')
print('=' * 72)
print(f"P50 MAE: {xgb_p50_metrics['MAE']:.4f}")
print(f"P90 Pinball: {xgb_p90_pinball:.4f}")
print(f"Coverage: {xgb_coverage*100:.2f}%")
print('TEST STATUS: UNTOUCHED')
print('Saved to:', OUTPUT_PATH)
print('=' * 72)


,Model,P50_MAE,P50_RMSE,P50_WAPE,P50_R2,P10_Pinball,P50_Pinball,P90_Pinball,Coverage_P10_P90,Mean_Interval_Width
0,Tuned XGBoost Quantile,95.010773,323.908323,0.531447,0.335009,15.820817,47.505387,36.53848,0.806743,267.769073


MADAD — XGBOOST VALIDATION CHECKPOINT
P50 MAE: 95.0108
P90 Pinball: 36.5385
Coverage: 80.67%
TEST STATUS: UNTOUCHED
Saved to: /content/drive/MyDrive/Madad for Medical Logistics/Dataset/Dryad_Dataset/forecasting_outputs_clean/03_xgboost_quantile
